<a href="https://colab.research.google.com/github/bainiao0706/Google-Drive-Remote-Upload/blob/main/GoogleDriveRemoteUpload.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#谷歌云盘的离线上传
#作者:bainiao0706

In [ ]:
from IPython.display import Javascript
Javascript('window.open("https://github.com/bainiao0706/Google-Drive-Remote-Upload");')

# 挂载Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# 离线上传文件

In [ ]:
!pip install tqdm --quiet
from concurrent.futures import ThreadPoolExecutor, as_completed
import requests
import os
from urllib.parse import urlparse, unquote
import threading
import time
import collections
from tqdm.auto import tqdm

# 下载目录
target_dir = '/content/drive/MyDrive/ColabDownloads' # 配置下载目录，默认下载会直接创建一个ColabDownloads的文件夹，编辑格式:/content/drive/MyDrive/你的文件夹名字
os.makedirs(target_dir, exist_ok=True) # 如果目录不存在则创建

# 下载任务列表
# 如果 'original_filename' 为空，则会从 URL 获取文件名。 'max_retries' 为下载失败后的最大重试次数。
download_tasks = [
    {
        "url": "",
        "original_filename": "",
        "max_retries": 3
    },
    {
        "url": "",
        "original_filename": "",
        "max_retries": 3
    },
    {
        "url": "",
        "original_filename": "",
        "max_retries": 3
    },
    # 您可以在这里添加更多下载任务，最大32个任务同时下载
]

# 下载文件的函数
def download_file(task):
    url = task['url']
    original_filename = task.get('original_filename', '')
    retry_attempt = task.get('_current_retry_attempt', 0) # 获取当前尝试次数

    if not url:
        # 如果 URL 为空，返回失败信息
        return {"status": "FAILED", "url": url, "error": "URL is empty.", "retry_attempt": retry_attempt, "filename": "Unknown"}

    # 如果原始文件名为空，则尝试从 URL 中提取
    if not original_filename:
        parsed_url = urlparse(url) # 解析 URL
        filename_from_url = os.path.basename(parsed_url.path) # 获取路径的最后一部分作为文件名
        original_filename = unquote(filename_from_url) # 解码 URL 编码的字符
        if not original_filename:
            original_filename = f"downloaded_file_{threading.get_ident()}.bin" # 如果URL中没有文件名，则使用通用名加线程ID作为备用
        print(f"[任务 {original_filename}] 从 URL 推断文件名: {url}")

    # 使用线程ID来创建临时的唯一文件名，防止不同线程同时下载时文件名冲突
    # 加上原始文件名的basename是为了避免不同URL但相同线程ID的冲突
    temp_filename = f"temp_{threading.get_ident()}_{os.path.basename(original_filename)}.part"
    temp_path = os.path.join(target_dir, temp_filename)
    final_path = os.path.join(target_dir, original_filename)

    start_byte = 0
    headers = {}

    if os.path.exists(temp_path):
        start_byte = os.path.getsize(temp_path)
        if start_byte > 0:
            print(f"[任务 {original_filename}] 发现未完成的下载文件 '{temp_filename}'，已下载 {start_byte} 字节。尝试从此处继续...")
            headers = {'Range': f'bytes={start_byte}-'}

    print(f"[任务 {original_filename}] [尝试 {retry_attempt + 1}] 开始下载: {url}")

    try:
        # 使用 requests 库下载文件，设置流式传输、允许重定向和超时
        with requests.get(url, stream=True, allow_redirects=True, timeout=60, headers=headers) as r: #请求超时时间默认为60s
            r.raise_for_status()  # 检查 HTTP 请求是否成功 (如 200 OK，否则抛出异常)

            # 确定文件总大小以及是续传还是重新开始
            total_size = 0 # 最终文件的总大小
            current_downloaded_bytes = start_byte # 当前已下载或将从此处开始下载的字节数

            file_mode = 'wb' # 默认以写入模式 (重新开始)

            if r.status_code == 206: # Partial Content - 服务器响应了 Range 头
                content_range = r.headers.get('Content-Range')
                if content_range:
                    # 示例: 'bytes 0-999/10000' -> total_size = 10000
                    total_size = int(content_range.split('/')[-1])
                else: # 如果 Content-Range 缺失但状态是 206，则退而求其次
                    total_size = start_byte + int(r.headers.get('Content-Length', 0))
                file_mode = 'ab' # 以追加模式写入现有文件
                print(f"[任务 {original_filename}] 服务器支持断点续传。从 {current_downloaded_bytes} 字节处继续...")
            elif r.status_code == 200: # OK - 服务器发送了完整文件，可能忽略了 Range 或这是一个全新下载
                total_size = int(r.headers.get('Content-Length', 0))
                if start_byte > 0: # 如果之前有下载但服务器未响应续传
                    print(f"[任务 {original_filename}] 服务器未响应断点续传请求，将重新开始下载...")
                current_downloaded_bytes = 0 # 重置初始进度，因为我们重新开始
                file_mode = 'wb' # 覆盖现有文件

            if total_size == 0: # 如果 Content-Length 或 Content-Range 不可用
                print(f"[任务 {original_filename}] 警告: 无法获取文件总大小，进度条可能不准确。")
                total_size = None # tqdm 可以处理 None 作为总大小

            # 使用 tqdm 显示进度条
            # position 参数用于为每个并发下载的进度条设置不同的行，防止重叠
            pos = (threading.get_ident() % MAX_CONCURRENT_DOWNLOADS) + 1
            with tqdm(
                initial=current_downloaded_bytes, # 从已下载的字节数开始进度条
                total=total_size, # 完整文件的总大小
                unit='B',
                unit_scale=True,
                desc=f"下载中 {original_filename} (并发: {MAX_CONCURRENT_DOWNLOADS})", # 进度条描述，增加并发线程数显示
                miniters=1,
                smoothing=0.1,
                position=pos,
                leave=True # 下载完成后保留进度条在屏幕上
            ) as pbar:
                with open(temp_path, file_mode) as f: # 使用计算出的模式 'wb' 或 'ab'
                    for chunk in r.iter_content(chunk_size=8192): # 每次读取 8KB 数据块
                        if chunk:
                            f.write(chunk)
                            pbar.update(len(chunk)) # 更新进度条

            # 最终检查：如果 total_size 为 None，尝试从最终文件大小获取
            if total_size is None:
                total_size = os.path.getsize(temp_path)

            os.rename(temp_path, final_path) # 下载完成后，将临时文件重命名为最终文件名
            print(f"[任务 {original_filename}] 成功下载并保存: {final_path}")
            # 返回字典格式的成功信息
            return {"status": "SUCCESS", "path": final_path, "url": url, "retry_attempt": retry_attempt, "filename": original_filename, "size": total_size}

    except requests.exceptions.RequestException as e:
        print(f"[任务 {original_filename}] 下载错误 (尝试 {retry_attempt + 1}): {e}")
        # 重要的: 不要在这里删除 temp_path，以便断点续传
        # if os.path.exists(temp_path):
        #     os.remove(temp_path) # 清理部分下载的文件
        # 返回字典格式的失败信息
        return {"status": "FAILED", "url": url, "error": str(e), "retry_attempt": retry_attempt, "filename": original_filename}
    except Exception as e:
        print(f"[任务 {original_filename}] 发生意外错误 (尝试 {retry_attempt + 1}): {e}")
        # 重要的: 不要在这里删除 temp_path，以便断点续传
        # if os.path.exists(temp_path):
        #     os.remove(temp_path) # 清理部分下载的文件
        # 返回字典格式的失败信息
        return {"status": "FAILED", "url": url, "error": str(e), "retry_attempt": retry_attempt, "filename": original_filename}

# 使用 ThreadPoolExecutor 管理并发下载
MAX_CONCURRENT_DOWNLOADS = 32 # 最大并发下载数量，默认32
final_results = []

# 准备任务队列，为每个任务添加内部状态
tasks_to_process = collections.deque() # 使用双端队列管理待处理的任务
for task in download_tasks:
    task_copy = task.copy() # 复制任务字典以添加内部状态
    task_copy['_current_retry_attempt'] = 0 # 初始化当前重试尝试次数
    tasks_to_process.append(task_copy)

active_futures = {} # 映射 future 对象到其对应的任务字典，用于跟踪活跃任务

with ThreadPoolExecutor(max_workers=MAX_CONCURRENT_DOWNLOADS) as executor:
    while tasks_to_process or active_futures:
        # 提交新任务，直到达到最大并发数或任务队列为空
        while tasks_to_process and len(active_futures) < MAX_CONCURRENT_DOWNLOADS:
            task = tasks_to_process.popleft() # 从队列头部取出一个任务
            future = executor.submit(download_file, task) # 提交任务到线程池
            active_futures[future] = task # 将 future 和任务关联起来

        if not active_futures and not tasks_to_process: # 如果没有活跃的 future 也没有待处理的任务，则退出循环
            break

        # 等待任何一个活跃的 future 完成
        # 移除 timeout 参数，使其阻塞直到有任务完成
        done_futures = as_completed(active_futures)

        for future in done_futures:
            task = active_futures.pop(future) # 从活跃任务中移除已完成的 future
            result = future.result() # 获取下载结果

            if result['status'] == "FAILED":
                current_retry_attempt = task['_current_retry_attempt']
                # 获取任务配置的最大重试次数，默认为0 (不重试)
                max_retries = task.get('max_retries', 0)

                if current_retry_attempt < max_retries:
                    task['_current_retry_attempt'] += 1 # 增加重试计数
                    # 指数退避策略: 2s, 4s, 8s, ... (2的n次方秒)
                    sleep_time = 2 ** task['_current_retry_attempt']
                    print(f"[任务 {result.get('filename', '未知文件')}] 重试下载 (尝试 {task['_current_retry_attempt'] + 1}/{max_retries + 1})，将在 {sleep_time} 秒后重试...")
                    time.sleep(sleep_time)
                    tasks_to_process.append(task) # 将任务重新添加到队列尾部，等待再次提交
                else:
                    print(f"[任务 {result.get('filename', '未知文件')}] 下载失败，已尝试 {current_retry_attempt + 1} 次。放弃下载。")
                    final_results.append(result) # 达到最大重试次数，标记为最终失败
            else:
                final_results.append(result) # 任务成功完成

print("\n--- 下载总结 ---")
for res in final_results:
    # 格式化输出结果
    size_info = f"{res.get('size', 'N/A')} 字节" if res.get('size') else 'N/A'
    print(f"状态: {res['status']}, 文件名: {res.get('filename', 'N/A')}, 大小: {size_info}, URL: {res['url']}")
    if res['status'] == 'FAILED':
        print(f"  错误: {res['error']}")
print("------------------------")